# 10_silver_to_gold
Refreshes the Gold star schema for one date from Silver: dimensions (date, game, publisher, developer), facts (daily snapshot, price changes, new listings) and the audit mart (pipeline runs). Idempotent through MERGE.
Parameters: date (YYYY-MM-DD, defaults to the latest loaded date).

In [0]:
import uuid
from datetime import datetime
from pyspark.sql import functions as F, Window
from pyspark.sql.types import StructType, StructField, StringType, LongType, TimestampType

dbutils.widgets.text("date", "")

SILVER = "workspace.steam"
GOLD = "workspace.gold"
LAYER = "Silver-to-Gold"
spark.sql(f"USE {GOLD}")

history = spark.table(f"{SILVER}.silver_steam_history")
BASELINE_DATE = str(history.agg(F.min("load_date")).first()[0])   # first full load
latest = str(history.agg(F.max("load_date")).first()[0])

run_date = dbutils.widgets.get("date").strip() or latest
datetime.strptime(run_date, "%Y-%m-%d")  # fails fast on a bad date
date_key = int(run_date.replace("-", ""))
print(f"run_date: {run_date} | date_key: {date_key} | baseline: {BASELINE_DATE} | latest: {latest}")
start = datetime.now()

**Helper Functions**

In [0]:
def merge(df, table, keys, compare=True):
    """Upsert df into a Gold table on the given keys.
    Updates a row only if a value actually changed, so a rerun gives 0 inserted, 0 updated."""
    df.createOrReplaceTempView("src")
    on = " AND ".join(f"t.{k} = s.{k}" for k in keys)
    cols = [c for c in df.columns if c not in keys and c != "load_timestamp"]
    changed = " OR ".join(f"NOT (t.{c} <=> s.{c})" for c in cols) or "false"
    matched = f"WHEN MATCHED AND ({changed}) THEN UPDATE SET *" if compare else ""
    res = spark.sql(f"""
        MERGE INTO {table} t USING src s ON {on}
        {matched}
        WHEN NOT MATCHED THEN INSERT *
    """).first()
    return int(res["num_inserted_rows"]), int(res["num_updated_rows"])

def first_name(col):
    """First listed name (e.g. 'Valve, Hidden Path' -> 'Valve'); blank -> 'Unknown'."""
    n = F.trim(F.split(F.col(col), ",")[0])
    return F.when(n.isNull() | (n == ""), F.lit("Unknown")).otherwise(n)

def name_key(name_col):
    """Stable surrogate key from a name; 'Unknown' -> -1."""
    return F.when(F.col(name_col) == "Unknown", F.lit(-1)).otherwise(F.xxhash64(F.col(name_col)))

stats = {}   # rows inserted / updated per table, for the log
print("helpers ready")

**Populating dim_date, It creates one row per calendar day from Oct 9 to Oct 24 (including days with no data, like Oct 12 and 20, which is normal for a date dimension)**

In [0]:
dates = spark.sql(f"""
    SELECT explode(sequence(DATE'{BASELINE_DATE}', DATE'{run_date}', INTERVAL 1 DAY)) AS full_date
""")

dim_date = dates.select(
    F.date_format("full_date", "yyyyMMdd").cast("int").alias("date_key"),
    "full_date",
    F.dayofmonth("full_date").alias("day_of_month"),
    F.date_format("full_date", "EEEE").alias("day_name"),
    ((F.dayofweek("full_date") + 5) % 7 + 1).alias("day_of_week"),   # 1 = Monday
    F.dayofweek("full_date").isin(1, 7).alias("is_weekend"),
    F.weekofyear("full_date").alias("week_of_year"),
    F.month("full_date").alias("month_num"),
    F.date_format("full_date", "MMMM").alias("month_name"),
    F.quarter("full_date").alias("quarter"),
    F.year("full_date").alias("year"),
    F.current_timestamp().alias("load_timestamp"))

stats["dim_date"] = merge(dim_date, "dim_date", ["date_key"])
print("dim_date (inserted, updated):", stats["dim_date"])
display(spark.table("dim_date").orderBy("date_key").limit(5))

**Populating dim_game,dim_publisher and dim_developer. Expect dim_game with about 82,350 rows, and publisher and developer each in the tens of thousands**

In [0]:
silver = (spark.table(f"{SILVER}.silver_steam")
          .withColumn("publisher_name", first_name("publisher"))
          .withColumn("developer_name", first_name("developer")))

first_seen = history.groupBy("appid").agg(F.min("load_date").alias("first_tracked_date"))

dim_game = (silver.join(first_seen, "appid", "left").select(
    F.col("appid").alias("game_key"), "appid",
    F.coalesce("name", F.lit("Unknown")).alias("game_name"),
    "is_free", "first_tracked_date",
    F.current_timestamp().alias("load_timestamp")))
stats["dim_game"] = merge(dim_game, "dim_game", ["game_key"])

dim_publisher = (silver.select("publisher_name").distinct()
                 .select(name_key("publisher_name").alias("publisher_key"), "publisher_name",
                         F.current_timestamp().alias("load_timestamp")))
stats["dim_publisher"] = merge(dim_publisher, "dim_publisher", ["publisher_key"])

dim_developer = (silver.select("developer_name").distinct()
                 .select(name_key("developer_name").alias("developer_key"), "developer_name",
                         F.current_timestamp().alias("load_timestamp")))
stats["dim_developer"] = merge(dim_developer, "dim_developer", ["developer_key"])

for t in ["dim_game", "dim_publisher", "dim_developer"]:
    print(t, "(inserted, updated):", stats[t], "| rows:", spark.table(t).count())

**Populating fact_game_daily_snapshot for one day only, whatever is the latest day**

In [0]:
loaded_dates = {str(r.load_date) for r in history.select("load_date").distinct().collect()}
if run_date not in loaded_dates:
    raise ValueError(f"{run_date} has no Silver data; choose a loaded date")

# each game's latest history row on or before run_date = its state on that day
w = Window.partitionBy("appid").orderBy(F.col("load_date").desc())
asof = (history.where(F.col("load_date") <= run_date)
        .withColumn("rn", F.row_number().over(w)).where("rn = 1").drop("rn"))

def price_tier(col):
    return (F.when(F.col(col) == 0, 1).when(F.col(col) < 5, 2).when(F.col(col) < 20, 3)
             .when(F.col(col) < 40, 4).when(F.col(col) >= 40, 5))

snapshot = (asof
    .withColumn("publisher_name", first_name("publisher"))
    .withColumn("developer_name", first_name("developer"))
    .select(
        F.lit(date_key).alias("date_key"),
        F.col("appid").alias("game_key"),
        name_key("publisher_name").alias("publisher_key"),
        name_key("developer_name").alias("developer_key"),
        price_tier("initialprice_usd").alias("price_tier_key"),
        (F.col("owners_max") / 1000).cast("int").alias("owners_band_key"),
        "price_usd", "initialprice_usd", "discount_pct",
        (F.col("discount_pct") > 0).alias("is_on_sale"),
        "positive", "negative", "total_reviews", "approval_rate", "ccu",
        "owners_min", "owners_max",
        ((F.col("owners_min") + F.col("owners_max")) / 2).cast("bigint").alias("owners_mid"),
        F.current_timestamp().alias("load_timestamp")))

stats["fact_game_daily_snapshot"] = merge(snapshot, "fact_game_daily_snapshot", ["date_key", "game_key"])
print("fact_game_daily_snapshot (inserted, updated):", stats["fact_game_daily_snapshot"])
display(spark.sql(f"""
    SELECT COUNT(*) AS games, SUM(CAST(is_on_sale AS INT)) AS on_sale, ROUND(AVG(discount_pct), 1) AS avg_discount
    FROM fact_game_daily_snapshot WHERE date_key = {date_key}
"""))

**Populating fact_price_change**

In [0]:
# rows for run_date, compared with each game's previous history row (any earlier date)
w_prev = Window.partitionBy("appid").orderBy("load_date")
pairs = (history.where(F.col("load_date") <= run_date)
         .withColumn("prev_price", F.lag("price_usd").over(w_prev))
         .withColumn("prev_initial", F.lag("initialprice_usd").over(w_prev))
         .withColumn("prev_discount", F.lag("discount_pct").over(w_prev))
         .withColumn("has_prev", F.lag("appid").over(w_prev).isNotNull())
         .where((F.col("load_date") == run_date) & F.col("has_prev")))

change_type = (
    F.when((F.col("prev_discount") == 0) & (F.col("discount_pct") > 0), 1)                 # sale started
     .when((F.col("prev_discount") > 0) & (F.col("discount_pct") == 0), 2)                 # sale ended
     .when((F.col("prev_discount") > 0) & (F.col("discount_pct") != F.col("prev_discount")), 3)  # discount changed
     .when(F.col("initialprice_usd") > F.col("prev_initial"), 4)                           # list price up
     .when(F.col("initialprice_usd") < F.col("prev_initial"), 5))                          # list price down

price_change = (pairs
    .withColumn("change_type_key", change_type)
    .where(F.col("change_type_key").isNotNull())
    .withColumn("publisher_name", first_name("publisher"))
    .withColumn("developer_name", first_name("developer"))
    .select(
        F.lit(date_key).alias("date_key"),
        F.col("appid").alias("game_key"),
        name_key("publisher_name").alias("publisher_key"),
        name_key("developer_name").alias("developer_key"),
        "change_type_key",
        F.col("prev_price").alias("old_price_usd"),
        F.col("price_usd").alias("new_price_usd"),
        F.round(F.col("price_usd") - F.col("prev_price"), 2).alias("price_change_usd"),
        F.col("prev_discount").alias("old_discount_pct"),
        F.col("discount_pct").alias("new_discount_pct"),
        F.col("prev_initial").alias("old_initialprice_usd"),
        F.col("initialprice_usd").alias("new_initialprice_usd"),
        F.current_timestamp().alias("load_timestamp")))

stats["fact_price_change"] = merge(price_change, "fact_price_change", ["date_key", "game_key"])
print("fact_price_change (inserted, updated):", stats["fact_price_change"])
display(spark.sql(f"""
    SELECT d.change_type, COUNT(*) AS events
    FROM fact_price_change f JOIN dim_change_type d USING (change_type_key)
    WHERE f.date_key = {date_key} GROUP BY d.change_type ORDER BY events DESC
"""))

**Populating fact_game_listing. It records the games that appeared for the first time on Oct 24, then shows 10 of them**

In [0]:
# games whose first ever history row is on run_date; the baseline full load is not a "new listing"
new_games = first_seen.where((F.col("first_tracked_date") == run_date) &
                             (F.col("first_tracked_date") > BASELINE_DATE))

listing = (new_games.join(snapshot.drop("date_key", "load_timestamp"),
                          new_games.appid == snapshot.game_key)
    .select(
        F.lit(date_key).alias("date_key"),
        "game_key", "publisher_key", "developer_key",
        "price_tier_key", "owners_band_key",
        F.current_timestamp().alias("load_timestamp")))

stats["fact_game_listing"] = merge(listing, "fact_game_listing", ["game_key"])
print("fact_game_listing (inserted, updated):", stats["fact_game_listing"])
display(spark.sql(f"""
    SELECT g.game_name, p.publisher_name, t.tier_name, o.band_label
    FROM fact_game_listing f
    JOIN dim_game g USING (game_key)
    JOIN dim_publisher p USING (publisher_key)
    JOIN dim_price_tier t USING (price_tier_key)
    JOIN dim_owners_band o USING (owners_band_key)
    WHERE f.date_key = {date_key}
    LIMIT 10
"""))

**Populating fact_pipeline_run (audit mart) for the given date**

In [0]:
logs = spark.table(f"{SILVER}.pipeline_execution_logs")
layers = spark.table("dim_pipeline_layer").select("layer_key", F.col("layer_name").alias("layer"))

pipeline_run = (logs.join(layers, "layer", "left")
    .select(
        "run_id",
        F.date_format("start_time", "yyyyMMdd").cast("int").alias("date_key"),
        "layer_key", "load_type", "status", "file_processed",
        "start_time", "end_time",
        F.round(F.col("end_time").cast("double") - F.col("start_time").cast("double"), 2)
         .alias("duration_seconds"),
        "rows_inserted", "rows_updated", "rows_skipped_unchanged", "rows_quarantined",
        "error_message",
        F.current_timestamp().alias("load_timestamp")))

# make sure every run date exists in dim_date (runs can fall on days outside the data range)
run_dates = (pipeline_run.select(F.to_date(F.col("date_key").cast("string"), "yyyyMMdd").alias("full_date"))
             .distinct())
extra_dates = run_dates.select(
    F.date_format("full_date", "yyyyMMdd").cast("int").alias("date_key"), "full_date",
    F.dayofmonth("full_date").alias("day_of_month"), F.date_format("full_date", "EEEE").alias("day_name"),
    ((F.dayofweek("full_date") + 5) % 7 + 1).alias("day_of_week"), F.dayofweek("full_date").isin(1, 7).alias("is_weekend"),
    F.weekofyear("full_date").alias("week_of_year"), F.month("full_date").alias("month_num"),
    F.date_format("full_date", "MMMM").alias("month_name"), F.quarter("full_date").alias("quarter"),
    F.year("full_date").alias("year"), F.current_timestamp().alias("load_timestamp"))
merge(extra_dates, "dim_date", ["date_key"])

stats["fact_pipeline_run"] = merge(pipeline_run, "fact_pipeline_run", ["run_id"])
print("fact_pipeline_run (inserted, updated):", stats["fact_pipeline_run"])
display(spark.sql("""
    SELECT l.layer_name, f.status, COUNT(*) AS runs, ROUND(AVG(f.duration_seconds), 1) AS avg_seconds
    FROM fact_pipeline_run f JOIN dim_pipeline_layer l USING (layer_key)
    GROUP BY 1, 2 ORDER BY 1, 2
"""))

In [0]:
LOG_SCHEMA = StructType([
    StructField("run_id", StringType()), StructField("layer", StringType()),
    StructField("load_type", StringType()), StructField("file_processed", StringType()),
    StructField("batch_id", StringType()), StructField("start_time", TimestampType()),
    StructField("end_time", TimestampType()), StructField("status", StringType()),
    StructField("rows_inserted", LongType()), StructField("rows_updated", LongType()),
    StructField("rows_skipped_unchanged", LongType()), StructField("rows_quarantined", LongType()),
    StructField("error_message", StringType())])

def write_log(status, inserted=None, updated=None, error=None):
    row = (str(uuid.uuid4()), LAYER, "incremental", f"gold date={run_date}", None,
           start, datetime.now(), status, inserted, updated, None, None, error)
    spark.createDataFrame([row], LOG_SCHEMA).write.mode("append") \
         .saveAsTable(f"{SILVER}.pipeline_execution_logs")

inserted = sum(v[0] for v in stats.values())
updated = sum(v[1] for v in stats.values())
write_log("Success", inserted, updated)

for t, (i, u) in stats.items():
    print(f"{t:28s} inserted {i:>7,}  updated {u:>7,}")
print(f"{'TOTAL':28s} inserted {inserted:>7,}  updated {updated:>7,}")